# Triangulum Galaxy Telescope Live-Stack Shorts Notebook

This notebook creates a **vertical 9:16 telescope-style live-stacking video** for YouTube Shorts.

Settings:
- Resolution: **720 × 1280**
- Length: **16.5 seconds**
- FPS: **18**
- Style: synthetic FITS/telescope data, stacking progress, HUD telemetry, reticles, scan sweep, and final reveal text

Run the cells from top to bottom. The notebook will save an `.mp4` video and a preview `.png` in the working folder.


In [ ]:
# Uncomment in a fresh environment if needed:
# %pip install -U numpy pillow imageio imageio-ffmpeg

In [1]:

from __future__ import annotations

from pathlib import Path
import math
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import imageio.v2 as imageio

# =============================
# YouTube Shorts video settings
# =============================
OUT_DIR = Path('.')
W, H = 720, 1280          # vertical 9:16 Shorts format
FPS = 18
SECONDS = 16.5            # at least 15 seconds
NFRAMES = int(FPS * SECONDS)

# Pick one galaxy here. Options:
# 'andromeda', 'triangulum', 'sombrero', 'pinwheel', 'black_eye', 'antennae'
GALAXY_ID = 'triangulum'

GALAXY_PRESETS = {
    'andromeda': {
        'title': 'M31 ANDROMEDA GALAXY',
        'subtitle': 'nearest major spiral • dust lanes + satellite glow',
        'target': 'M31 / Andromeda Galaxy',
        'final': 'DUST LANES REVEALED',
        'details': 'wide tilted spiral grows out of noisy telescope data',
        'seed': 3101,
        'style': 'andromeda',
        'center': (0.50, 0.52),
        'reticles': [(0.50, 0.52, 132), (0.63, 0.42, 38), (0.30, 0.66, 34)],
        'note': 'Synthetic FITS-style exposures for a 9:16 YouTube Short',
    },
    'triangulum': {
        'title': 'M33 TRIANGULUM GALAXY',
        'subtitle': 'face-on spiral • blue star-forming regions',
        'target': 'M33 / Triangulum Galaxy',
        'final': 'BLUE STAR CLOUDS EMERGE',
        'details': 'spiral arms sharpen as the live stack builds',
        'seed': 3303,
        'style': 'face_spiral',
        'arms': 2,
        'twist': 4.9,
        'scale': 0.32,
        'center': (0.50, 0.51),
        'reticles': [(0.50, 0.51, 105)],
        'note': 'Synthetic telescope stack with animated SNR gain',
    },
    'sombrero': {
        'title': 'M104 SOMBRERO GALAXY',
        'subtitle': 'edge-on galaxy • bright core + dark dust lane',
        'target': 'M104 / Sombrero Galaxy',
        'final': 'DUST RING LOCKED',
        'details': 'thin shadow band cuts across the glowing core',
        'seed': 1040,
        'style': 'sombrero',
        'center': (0.50, 0.51),
        'reticles': [(0.50, 0.51, 116)],
        'note': 'Synthetic deep-sky telescope acquisition sequence',
    },
    'pinwheel': {
        'title': 'M101 PINWHEEL GALAXY',
        'subtitle': 'wide spiral • asymmetric arms + H-II knots',
        'target': 'M101 / Pinwheel Galaxy',
        'final': 'SPIRAL ARMS UNFOLD',
        'details': 'faint outer structure appears after stacking',
        'seed': 1011,
        'style': 'face_spiral',
        'arms': 3,
        'twist': 5.7,
        'scale': 0.36,
        'center': (0.50, 0.52),
        'reticles': [(0.50, 0.52, 122)],
        'note': 'Synthetic FITS-style spiral galaxy data animation',
    },
    'black_eye': {
        'title': 'M64 BLACK EYE GALAXY',
        'subtitle': 'dark dust feature • glowing galactic core',
        'target': 'M64 / Black Eye Galaxy',
        'final': 'DARK EYE FEATURE FOUND',
        'details': 'contrast rises until the dust patch stands out',
        'seed': 6400,
        'style': 'black_eye',
        'center': (0.50, 0.51),
        'reticles': [(0.50, 0.51, 105), (0.43, 0.48, 32)],
        'note': 'Synthetic telescope stack for a mysterious deep-sky Short',
    },
    'antennae': {
        'title': 'ANTENNAE GALAXIES',
        'subtitle': 'colliding galaxies • tidal tails + starburst knots',
        'target': 'NGC 4038 / NGC 4039',
        'final': 'TIDAL TAILS DETECTED',
        'details': 'two galaxies collide as faint arcs rise from noise',
        'seed': 4038,
        'style': 'antennae',
        'center': (0.50, 0.52),
        'reticles': [(0.45, 0.50, 62), (0.56, 0.55, 62), (0.50, 0.52, 128)],
        'note': 'Synthetic galaxy interaction live-stack animation',
    },
}

PRESET = GALAXY_PRESETS[GALAXY_ID]
SEED = int(PRESET['seed'])
rng = np.random.default_rng(SEED)
SLUG = GALAXY_ID.replace('_', '-')
VIDEO_PATH = OUT_DIR / f'{SLUG}_galaxy_telescope_live_stack_shorts.mp4'
PNG_PATH = OUT_DIR / f'{SLUG}_galaxy_shorts_preview.png'


def robust_scale(arr, lo=0.25, hi=99.85, gamma=0.72):
    arr = np.asarray(arr, dtype=np.float32)
    a, b = np.percentile(arr, [lo, hi])
    scaled = np.clip((arr - a) / max(b - a, 1e-6), 0, 1)
    return scaled ** gamma


def smooth_noise(width=W, height=H, seed=SEED, low=90, scale=1.0):
    local = np.random.default_rng(seed)
    small = local.random((max(8, height // low), max(8, width // low))).astype(np.float32)
    im = Image.fromarray((small * 255).astype(np.uint8), 'L').resize((width, height), Image.Resampling.BICUBIC)
    im = im.filter(ImageFilter.GaussianBlur(radius=10))
    arr = np.asarray(im).astype(np.float32) / 255.0
    return scale * arr


def add_gaussian(blob, x, y, amp, sx, sy=None):
    if sy is None:
        sy = sx
    height, width = blob.shape
    r = int(max(3, math.ceil(4 * max(sx, sy))))
    x0, x1 = max(0, int(x)-r), min(width, int(x)+r+1)
    y0, y1 = max(0, int(y)-r), min(height, int(y)+r+1)
    if x1 <= x0 or y1 <= y0:
        return
    yy, xx = np.mgrid[y0:y1, x0:x1]
    blob[y0:y1, x0:x1] += amp * np.exp(-(((xx-x)/sx)**2 + ((yy-y)/sy)**2) / 2)


def make_starfield(width=W, height=H, seed=SEED):
    local = np.random.default_rng(seed)
    img = np.zeros((height, width), dtype=np.float32)
    yy_full, xx_full = np.mgrid[0:height, 0:width]
    nstars = 2500
    xs = local.uniform(0, width, nstars)
    ys = local.uniform(0, height, nstars)
    flux = 0.04 + 1.35 * (1 - local.power(3.5, nstars)) ** 2
    sigma = local.uniform(0.35, 1.12, nstars)
    bright_idx = local.choice(nstars, size=40, replace=False)
    flux[bright_idx] *= local.uniform(1.7, 3.5, len(bright_idx))
    sigma[bright_idx] *= local.uniform(1.0, 1.85, len(bright_idx))
    for x, y, f, s in zip(xs, ys, flux, sigma):
        r = int(max(2, math.ceil(4 * s)))
        x0, x1 = max(0, int(x)-r), min(width, int(x)+r+1)
        y0, y1 = max(0, int(y)-r), min(height, int(y)+r+1)
        xx = xx_full[y0:y1, x0:x1]
        yy = yy_full[y0:y1, x0:x1]
        img[y0:y1, x0:x1] += f * np.exp(-((xx-x)**2 + (yy-y)**2) / (2*s*s))
    return img


def screen_background(x, y, width=W, height=H, seed=SEED):
    grad = 0.012 + 0.028*(1 - y/height) + 0.010*np.sin(2*np.pi*x/width)
    wisp = 0.030 * smooth_noise(width, height, seed+22, low=50, scale=1.0)
    return grad + wisp


def colorize(total, warm=None, blue=None, red=None, stars=None, dust=None):
    norm = robust_scale(total, 0.20, 99.93, gamma=0.76)
    warm_n = robust_scale(warm if warm is not None else total, 0.1, 99.80, gamma=0.74)
    blue_n = robust_scale(blue if blue is not None else np.zeros_like(total), 35, 99.80, gamma=0.72)
    red_n = robust_scale(red if red is not None else np.zeros_like(total), 35, 99.85, gamma=0.70)
    star_n = robust_scale(stars if stars is not None else np.zeros_like(total), 72, 99.98, gamma=0.62)
    dust_n = np.clip(dust if dust is not None else np.zeros_like(total), 0, 1)
    rgb = np.zeros((*total.shape, 3), dtype=np.float32)
    rgb[..., 0] = 0.55*norm + 0.32*warm_n + 0.18*red_n + 0.10*star_n
    rgb[..., 1] = 0.50*norm + 0.27*warm_n + 0.09*blue_n + 0.12*star_n
    rgb[..., 2] = 0.62*norm + 0.18*warm_n + 0.36*blue_n + 0.14*star_n
    rgb *= (1 - 0.18*dust_n[..., None])
    return np.clip(rgb, 0, 1)


def add_vignette(rgb):
    height, width = rgb.shape[:2]
    y, x = np.mgrid[0:height, 0:width]
    rr = np.sqrt(((x-width/2)/(width/2))**2 + ((y-height/2)/(height/2))**2)
    rgb = rgb * np.clip(1 - 0.48 * rr**1.55, 0.32, 1.0)[..., None]
    return np.clip(rgb, 0, 1)


def make_andromeda(width=W, height=H, seed=SEED):
    local = np.random.default_rng(seed)
    y, x = np.mgrid[0:height, 0:width]
    cx, cy = width*0.50, height*0.52
    theta = np.deg2rad(-25)
    xr = (x - cx) * np.cos(theta) - (y - cy) * np.sin(theta)
    yr = (x - cx) * np.sin(theta) + (y - cy) * np.cos(theta)
    a = width*0.56
    q = 0.18
    r = np.sqrt((xr/a)**2 + (yr/(a*q))**2)
    disk = 0.88*np.exp(-2.1*r)
    bulge = 2.2*np.exp(-((xr/(width*0.12))**2 + (yr/(width*0.050))**2))
    halo = 0.36*np.exp(-1.0*r)
    # Multiple curved dark lanes across the inclined disk.
    n = smooth_noise(width, height, seed+12, low=70)
    lane1 = np.exp(-((yr + 0.038*width*np.sin(xr/(width*0.10))) / (width*0.018))**2) * np.exp(-(xr/(width*0.55))**2)
    lane2 = np.exp(-((yr - width*0.055 + 0.025*width*np.sin(xr/(width*0.12)+1.3)) / (width*0.017))**2) * np.exp(-(xr/(width*0.50))**2)
    lane3 = np.exp(-((yr + width*0.095 + 0.022*width*np.sin(xr/(width*0.15)-0.8)) / (width*0.022))**2) * np.exp(-(xr/(width*0.48))**2)
    dust = np.clip((0.54*lane1 + 0.42*lane2 + 0.28*lane3)*(0.55 + n), 0, 0.72)
    galaxy = (disk + bulge + halo) * (1 - dust)
    # Satellite galaxies M32/M110 style glows.
    sat = np.zeros((height, width), dtype=np.float32)
    add_gaussian(sat, width*0.63, height*0.42, 0.85, width*0.032, width*0.025)
    add_gaussian(sat, width*0.30, height*0.66, 0.55, width*0.052, width*0.030)
    blue = np.zeros((height, width), dtype=np.float32)
    red = np.zeros((height, width), dtype=np.float32)
    for _ in range(130):
        xx = local.normal(cx, width*0.22)
        yy = cy + local.normal(0, height*0.045) + 0.04*(xx-cx)*math.sin(theta)
        amp = local.uniform(0.06, 0.18)
        sz = local.uniform(1.1, 2.8)
        add_gaussian(blue, xx, yy, amp, sz)
        if local.random() < 0.28:
            add_gaussian(red, xx+local.normal(0,2), yy+local.normal(0,2), amp*0.5, sz*1.6)
    stars = make_starfield(width, height, seed+5)
    bg = screen_background(x, y, width, height, seed)
    total = galaxy + sat + stars*0.70 + bg
    return add_vignette(colorize(total, warm=galaxy+sat, blue=blue, red=red, stars=stars, dust=dust))


def make_face_spiral(width=W, height=H, seed=SEED, arms=2, twist=5.0, scale_factor=0.33):
    local = np.random.default_rng(seed)
    y, x = np.mgrid[0:height, 0:width]
    cx, cy = width*0.50, height*0.52
    theta = np.deg2rad(local.uniform(-25, 25))
    xr = (x - cx) * np.cos(theta) - (y - cy) * np.sin(theta)
    yr = (x - cx) * np.sin(theta) + (y - cy) * np.cos(theta)
    scale = width*scale_factor
    r = np.sqrt((xr/scale)**2 + (yr/scale)**2)
    phi = np.arctan2(yr, xr)
    disk = 0.82*np.exp(-1.75*r)
    core = 1.55*np.exp(-((xr/(width*0.055))**2 + (yr/(width*0.055))**2))
    arm_img = np.zeros_like(r)
    dust = np.zeros_like(r)
    blue = np.zeros_like(r)
    red = np.zeros_like(r)
    for k in range(arms):
        phase = arms*phi + twist*r + k*(2*np.pi/arms)
        arm_img += np.exp(-(np.sin(phase)**2) / (0.034 + 0.028*r)) * np.exp(-0.86*r)
        dust += np.exp(-(np.sin(phase+0.45)**2) / (0.022 + 0.026*r)) * np.exp(-1.05*r)
    arms_luma = 0.85*arm_img
    dust *= 0.22 * (0.8 + smooth_noise(width, height, seed+11, low=65))
    # Star-forming knots along arms.
    for branch in range(arms):
        for _ in range(90):
            rr = local.uniform(0.23, 2.15)
            ph = -(twist/arms) * rr + branch*(2*math.pi/arms) + local.normal(0, 0.13)
            gx = rr * scale * math.cos(ph)
            gy = rr * scale * math.sin(ph)
            wx = cx + gx * math.cos(theta) + gy * math.sin(theta)
            wy = cy - gx * math.sin(theta) + gy * math.cos(theta)
            if 0 <= wx < width and 0 <= wy < height:
                amp = local.uniform(0.09, 0.33) * math.exp(-0.13*rr)
                sz = local.uniform(1.0, 3.1)
                add_gaussian(blue, wx, wy, amp, sz)
                if local.random() < 0.38:
                    add_gaussian(red, wx+local.normal(0,2), wy+local.normal(0,2), amp*0.52, sz*1.45)
    stars = make_starfield(width, height, seed+5)
    bg = screen_background(x, y, width, height, seed)
    galaxy = np.clip(disk + core + arms_luma, 0, None) * np.clip(1-dust, 0.32, 1.0)
    total = galaxy + stars*0.70 + bg
    return add_vignette(colorize(total, warm=galaxy, blue=blue, red=red, stars=stars, dust=dust))


def make_sombrero(width=W, height=H, seed=SEED):
    y, x = np.mgrid[0:height, 0:width]
    cx, cy = width*0.50, height*0.51
    theta = np.deg2rad(-5)
    xr = (x - cx) * np.cos(theta) - (y - cy) * np.sin(theta)
    yr = (x - cx) * np.sin(theta) + (y - cy) * np.cos(theta)
    halo = 0.68*np.exp(-np.sqrt((xr/(width*0.43))**2 + (yr/(width*0.13))**2))
    bulge = 2.25*np.exp(-((xr/(width*0.13))**2 + (yr/(width*0.095))**2))
    disk = 0.58*np.exp(-np.abs(yr)/(width*0.020))*np.exp(-(xr/(width*0.55))**2)
    lane = np.exp(-((yr + width*0.012*np.sin(xr/(width*0.12))) / (width*0.016))**2) * np.exp(-(xr/(width*0.53))**2)
    dust = np.clip(0.82*lane*(0.7 + smooth_noise(width, height, seed+13, low=78)), 0, 0.86)
    galaxy = (halo + bulge + disk)*(1-dust)
    stars = make_starfield(width, height, seed+5)
    bg = screen_background(x, y, width, height, seed)
    total = galaxy + stars*0.70 + bg
    blue = 0.05*disk
    red = 0.10*bulge
    return add_vignette(colorize(total, warm=galaxy, blue=blue, red=red, stars=stars, dust=dust))


def make_black_eye(width=W, height=H, seed=SEED):
    y, x = np.mgrid[0:height, 0:width]
    cx, cy = width*0.50, height*0.51
    theta = np.deg2rad(-18)
    xr = (x - cx) * np.cos(theta) - (y - cy) * np.sin(theta)
    yr = (x - cx) * np.sin(theta) + (y - cy) * np.cos(theta)
    r = np.sqrt((xr/(width*0.26))**2 + (yr/(width*0.17))**2)
    disk = 1.05*np.exp(-1.65*r)
    core = 2.05*np.exp(-((xr/(width*0.055))**2 + (yr/(width*0.048))**2))
    eye_x = xr + width*0.055
    eye_y = yr - width*0.020
    eye = np.exp(-((eye_x/(width*0.095))**2 + (eye_y/(width*0.036))**2))
    crescent = np.exp(-(((xr+width*0.030)/(width*0.14))**2 + ((yr-width*0.025)/(width*0.070))**2))
    dust = np.clip(0.90*eye + 0.34*crescent*smooth_noise(width,height,seed+7,low=62), 0, 0.92)
    galaxy = (disk + core)*(1-dust)
    blue = 0.12*disk*np.exp(-0.8*r)
    red = 0.11*core
    stars = make_starfield(width, height, seed+5)
    bg = screen_background(x, y, width, height, seed)
    total = galaxy + stars*0.70 + bg
    return add_vignette(colorize(total, warm=galaxy, blue=blue, red=red, stars=stars, dust=dust))


def make_antennae(width=W, height=H, seed=SEED):
    local = np.random.default_rng(seed)
    y, x = np.mgrid[0:height, 0:width]
    # Two colliding cores.
    c1x, c1y = width*0.45, height*0.50
    c2x, c2y = width*0.56, height*0.55
    core1 = 1.25*np.exp(-(((x-c1x)/(width*0.075))**2 + ((y-c1y)/(width*0.060))**2))
    core2 = 1.15*np.exp(-(((x-c2x)/(width*0.080))**2 + ((y-c2y)/(width*0.065))**2))
    envelope = 0.55*np.exp(-(((x-width*0.50)/(width*0.22))**2 + ((y-height*0.53)/(width*0.18))**2))
    # Long tidal arcs.
    def arc(cx, cy, rad, start, stop, amp=0.45, thick=0.026):
        ang = np.arctan2(y-cy, x-cx)
        rr = np.sqrt((x-cx)**2 + (y-cy)**2)
        # normalize angle interval around given span
        mid = (start+stop)/2
        span = abs(stop-start)/2
        dang = np.angle(np.exp(1j*(ang-mid)))
        mask = np.exp(-(dang/span)**8)
        return amp*np.exp(-((rr-rad)/(width*thick))**2)*mask
    tail1 = arc(width*0.45, height*0.52, width*0.39, -2.65, -0.42, amp=0.42, thick=0.025)
    tail2 = arc(width*0.53, height*0.51, width*0.43, 0.15, 2.38, amp=0.39, thick=0.025)
    bridge = 0.40*np.exp(-(((y-(height*0.52 + 0.22*(x-width*0.50))) / (width*0.045))**2 + ((x-width*0.50)/(width*0.16))**2))
    base = core1 + core2 + envelope + tail1 + tail2 + bridge
    dust = 0.26*smooth_noise(width,height,seed+9,low=70)*np.clip(base,0,1)
    blue = np.zeros((height, width), dtype=np.float32)
    red = np.zeros((height, width), dtype=np.float32)
    for _ in range(170):
        if local.random() < 0.65:
            wx = local.normal(width*0.50, width*0.12)
            wy = local.normal(height*0.53, width*0.10)
        else:
            # knots along the tails
            t = local.uniform(0, 1)
            ang = local.choice([local.uniform(-2.6,-0.4), local.uniform(0.1,2.4)])
            rr = width*local.uniform(0.26,0.45)
            wx = width*0.50 + rr*math.cos(ang)
            wy = height*0.52 + rr*math.sin(ang)
        if 0 <= wx < width and 0 <= wy < height:
            amp = local.uniform(0.08, 0.34)
            sz = local.uniform(1.1, 3.4)
            add_gaussian(blue, wx, wy, amp, sz)
            if local.random() < 0.55:
                add_gaussian(red, wx+local.normal(0,2), wy+local.normal(0,2), amp*0.55, sz*1.55)
    stars = make_starfield(width, height, seed+5)
    bg = screen_background(x, y, width, height, seed)
    total = base*(1-dust) + stars*0.70 + bg
    return add_vignette(colorize(total, warm=base, blue=blue, red=red, stars=stars, dust=dust))


def make_galaxy_base(preset=PRESET, width=W, height=H):
    style = preset['style']
    if style == 'andromeda':
        return make_andromeda(width, height, int(preset['seed']))
    if style == 'face_spiral':
        return make_face_spiral(width, height, int(preset['seed']), preset.get('arms', 2), preset.get('twist', 5.0), preset.get('scale', 0.33))
    if style == 'sombrero':
        return make_sombrero(width, height, int(preset['seed']))
    if style == 'black_eye':
        return make_black_eye(width, height, int(preset['seed']))
    if style == 'antennae':
        return make_antennae(width, height, int(preset['seed']))
    raise ValueError(f'Unknown style: {style}')


def fonts():
    try:
        return {
            'title': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 30),
            'big': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 25),
            'body': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18),
            'small': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 15),
            'mono': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf', 16),
        }
    except Exception:
        return {'title': None, 'big': None, 'body': None, 'small': None, 'mono': None}

FONT_CACHE = fonts()


def draw_reticle(draw, cx, cy, r, alpha=105):
    col = (210, 230, 255, alpha)
    draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline=col, width=2)
    draw.line([cx-r-14, cy, cx-r+8, cy], fill=col, width=2)
    draw.line([cx+r-8, cy, cx+r+14, cy], fill=col, width=2)
    draw.line([cx, cy-r-14, cx, cy-r+8], fill=col, width=2)
    draw.line([cx, cy+r-8, cx, cy+r+14], fill=col, width=2)


def add_cosmic_rays(arr, frame_index, progress):
    if progress > 0.42:
        return arr
    local = np.random.default_rng(SEED + frame_index * 19)
    out = arr.copy()
    h, w, _ = out.shape
    n = 2 + int(4*(1-progress))
    im = Image.fromarray((out*255).astype(np.uint8), 'RGB')
    d = ImageDraw.Draw(im, 'RGBA')
    for _ in range(n):
        x = int(local.integers(30, w-30))
        y = int(local.integers(100, h-120))
        length = int(local.integers(18, 65))
        angle = local.uniform(-1.0, 1.0)
        dx = int(math.cos(angle) * length)
        dy = int(math.sin(angle) * length)
        a = int(60 + 95*(1-progress))
        d.line([x, y, x+dx, y+dy], fill=(230, 245, 255, a), width=int(local.integers(1, 3)))
    return np.asarray(im).astype(np.float32)/255.0


def add_overlay(frame_rgb, frame_index, total_frames, preset=PRESET, width=W, height=H):
    im = Image.fromarray(np.clip(frame_rgb * 255, 0, 255).astype(np.uint8), 'RGB')
    draw = ImageDraw.Draw(im, 'RGBA')
    f = FONT_CACHE
    p = frame_index / max(total_frames-1, 1)
    n_exposures = 1 + int(p * 144)
    total_seconds = n_exposures * 25
    snr = math.sqrt(n_exposures) * 7.8
    seeing = 1.85 - 0.42*p + 0.06*math.sin(frame_index*0.22)
    sky = 21.08 + 0.18*math.sin(frame_index*0.09 + 1.0)

    draw.rounded_rectangle([18, 18, width-18, 126], radius=18, fill=(3, 7, 18, 162), outline=(170,190,220,88), width=1)
    draw.text((34, 34), preset['title'], fill=(238, 242, 250, 248), font=f['title'])
    draw.text((36, 72), 'live telescope stack • ' + preset['subtitle'], fill=(191, 206, 225, 232), font=f['body'])
    draw.text((36, 98), preset.get('note', 'Synthetic FITS-style exposures for a 9:16 YouTube Short'), fill=(170, 190, 214, 220), font=f['small'])

    # Targeting reticles.
    reticles = preset.get('reticles', [(0.5, 0.5, 100)])
    for idx, (rx, ry, rr) in enumerate(reticles):
        pulse = 4*math.sin(frame_index*0.08 + idx)
        draw_reticle(draw, width*rx, height*ry, rr + int(pulse), alpha=88 if idx == 0 else 72)
    if len(reticles) >= 2:
        x0, y0, _ = reticles[0]
        for rx, ry, _ in reticles[1:]:
            draw.line([width*x0, height*y0, width*rx, height*ry], fill=(170,205,255,38), width=2)

    panel_x0, panel_y0, panel_x1, panel_y1 = 24, height-314, width-24, height-132
    draw.rounded_rectangle([panel_x0, panel_y0, panel_x1, panel_y1], radius=16, fill=(4, 9, 20, 165), outline=(175,198,225,95), width=1)
    rows = [
        ('Target', preset['target']),
        ('Frame', f'{frame_index+1:03d}/{total_frames:03d}'),
        ('Stacked', f'{n_exposures:03d} × 25 s  ({total_seconds/60:4.1f} min)'),
        ('Seeing', f'{seeing:0.2f} arcsec'),
        ('Sky', f'{sky:0.2f} mag/arcsec²'),
        ('SNR', f'{snr:0.1f}'),
    ]
    y0 = panel_y0 + 18
    for k, v in rows:
        draw.text((panel_x0+18, y0), k, fill=(154, 178, 205, 232), font=f['small'])
        draw.text((panel_x0+126, y0), v, fill=(234, 240, 250, 245), font=f['mono'])
        y0 += 26

    bx0, by0, bx1, by1 = 24, height-86, width-24, height-52
    draw.text((24, height-118), 'RAW NOISE  →  STACKED SIGNAL', fill=(224, 233, 245, 236), font=f['body'])
    draw.rounded_rectangle([bx0, by0, bx1, by1], radius=12, fill=(5, 10, 20, 175), outline=(170,190,220,100), width=1)
    draw.rounded_rectangle([bx0+4, by0+4, bx0+4+int((bx1-bx0-8)*p), by1-4], radius=10, fill=(226, 236, 255, 162))

    sweep_x = int(width * ((frame_index % FPS) / FPS))
    draw.rectangle([sweep_x, 134, min(width, sweep_x+3), height-130], fill=(210, 230, 255, 42))
    if p > 0.78:
        alpha = int(min(210, (p-0.78)/0.22*210))
        draw.rounded_rectangle([50, 152, width-50, 226], radius=16, fill=(5, 11, 22, 120), outline=(210,230,255,70), width=1)
        draw.text((72, 170), preset['final'], fill=(240, 246, 255, alpha), font=f['big'])
        draw.text((74, 202), preset['details'], fill=(192, 208, 228, alpha), font=f['small'])
    return np.asarray(im)


def render_video(galaxy_id=GALAXY_ID):
    global PRESET, SEED, rng, SLUG, VIDEO_PATH, PNG_PATH
    PRESET = GALAXY_PRESETS[galaxy_id]
    SEED = int(PRESET['seed'])
    rng = np.random.default_rng(SEED)
    SLUG = galaxy_id.replace('_', '-')
    VIDEO_PATH = OUT_DIR / f'{SLUG}_galaxy_telescope_live_stack_shorts.mp4'
    PNG_PATH = OUT_DIR / f'{SLUG}_galaxy_shorts_preview.png'

    base = make_galaxy_base(PRESET)
    Image.fromarray((base*255).astype(np.uint8)).save(PNG_PATH)
    writer = imageio.get_writer(
        VIDEO_PATH,
        fps=FPS,
        codec='libx264',
        quality=8,
        macro_block_size=16,
        output_params=['-pix_fmt', 'yuv420p', '-crf', '23', '-movflags', '+faststart']
    )
    try:
        for i in range(NFRAMES):
            p = i / max(NFRAMES-1, 1)
            nstack = 1 + int(p * 144)
            noise_sigma = 0.095 / math.sqrt(nstack) + 0.0045
            frame = np.clip(base + rng.normal(0, noise_sigma, base.shape).astype(np.float32), 0, 1)
            frame = np.clip((frame - 0.018) * (0.86 + 0.22*p), 0, 1)
            frame = add_cosmic_rays(frame, i, p)
            img = Image.fromarray((frame*255).astype(np.uint8), 'RGB')
            blur_radius = max(0.0, 0.95*(1-p)**1.8)
            if blur_radius > 0.03:
                img = img.filter(ImageFilter.GaussianBlur(radius=blur_radius))

            # Slow phone-friendly push-in and drift.
            cx_frac, cy_frac = PRESET.get('center', (0.50, 0.52))
            zoom = 1.00 + 0.105*p + 0.012*math.sin(2*math.pi*p*1.5)
            crop_w, crop_h = int(W / zoom), int(H / zoom)
            center_x = W*(cx_frac + 0.026*math.sin(2*math.pi*p))
            center_y = H*(cy_frac - 0.030*math.cos(2*math.pi*p*0.7))
            left = int(np.clip(center_x - crop_w/2, 0, W-crop_w))
            top = int(np.clip(center_y - crop_h/2, 0, H-crop_h))
            crop = img.crop((left, top, left+crop_w, top+crop_h)).resize((W, H), Image.Resampling.LANCZOS)
            frame = np.asarray(crop).astype(np.float32) / 255.0
            writer.append_data(add_overlay(frame, i, NFRAMES, PRESET))
    finally:
        writer.close()
    print('Saved preview:', PNG_PATH.resolve())
    print('Saved video:', VIDEO_PATH.resolve())
    return VIDEO_PATH


# Render the selected galaxy. To render a different one, change GALAXY_ID above or call render_video('sombrero').
render_video(GALAXY_ID)


Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


Saved preview: /home/jatin/Downloads/triangulum_galaxy_shorts_preview.png
Saved video: /home/jatin/Downloads/triangulum_galaxy_telescope_live_stack_shorts.mp4


PosixPath('triangulum_galaxy_telescope_live_stack_shorts.mp4')

## Optional: use real telescope/FITS frames instead of the synthetic galaxy

This notebook uses synthetic deep-sky imagery so it runs anywhere. To adapt it for real telescope data, replace the `base + noise` section inside `render_video()` with a running stack from aligned calibrated frames. For example, if you have a NumPy array named `data_cube` with shape `(number_of_exposures, height, width)`, normalize each cumulative average and send it through the overlay step.
